<a href="https://colab.research.google.com/github/Santiago-Echeverri-Arteaga/Fisica_Computacional_2/blob/master/curso_2026_2/04_vision/43_transfer_learning_fisica.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg"
       alt="Abrir en Colab"/>
</a>

# Transfer learning con modelos ImageNet

**Pregunta guía:** ¿Cuándo reutilizar representaciones visuales ayuda a un problema nuevo?<br>
**Duración sugerida:** 4 horas.<br>
**Entorno:** CPU; datos incluidos o generados en memoria.

El orden de trabajo es siempre: problema → matemática → implementación
mínima → biblioteca → evaluación → interpretación física.


**Requiere TensorFlow e Internet para descargar CIFAR-10 y pesos.** El
flujo tiene dos fases: congelar una base preentrenada y ajustar la cabeza;
luego, opcionalmente, descongelar pocas capas con tasa muy pequeña. El
test se mantiene cerrado. MobileNetV2 permite una práctica rápida; cambie
una constante para usar InceptionV3.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.metrics import accuracy_score
from tensorflow import keras
from tensorflow.keras import layers

SEMILLA=42
keras.utils.set_random_seed(SEMILLA)
ARQUITECTURA="MobileNetV2"  # cambie a "InceptionV3" para el laboratorio extendido
(X_train,y_train),(X_test,y_test)=keras.datasets.cifar10.load_data()
y_train,y_test=y_train.ravel(),y_test.ravel()
clases=[0,1,2]  # avión, automóvil, ave: ejemplo pequeño
mask_train=np.isin(y_train,clases); mask_test=np.isin(y_test,clases)
X_dev,y_dev=X_train[mask_train][:4500],y_train[mask_train][:4500]
X_test,y_test=X_test[mask_test][:1200],y_test[mask_test][:1200]
orden=np.random.default_rng(SEMILLA).permutation(len(X_dev))
corte=int(.8*len(orden)); tr,val=orden[:corte],orden[corte:]


In [ ]:
if ARQUITECTURA=="InceptionV3":
    tamaño=(96,96); Base=keras.applications.InceptionV3; preprocesar=keras.applications.inception_v3.preprocess_input
else:
    tamaño=(96,96); Base=keras.applications.MobileNetV2; preprocesar=keras.applications.mobilenet_v2.preprocess_input

aumentación=keras.Sequential([
    layers.RandomFlip("horizontal"), layers.RandomRotation(.05),
],name="aumentación")
base=Base(weights="imagenet",include_top=False,input_shape=(*tamaño,3))
base.trainable=False
entrada=keras.Input((32,32,3))
x=layers.Resizing(*tamaño)(entrada); x=aumentación(x); x=preprocesar(x)
x=base(x,training=False); x=layers.GlobalAveragePooling2D()(x); x=layers.Dropout(.25)(x)
salida=layers.Dense(len(clases))(x)
modelo=keras.Model(entrada,salida)
modelo.compile(optimizer=keras.optimizers.Adam(1e-3),loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),metrics=["accuracy"])
modelo.fit(X_dev[tr],y_dev[tr],validation_data=(X_dev[val],y_dev[val]),epochs=5,batch_size=64,verbose=2,
           callbacks=[keras.callbacks.EarlyStopping(patience=2,restore_best_weights=True)])


In [ ]:
# Fine-tuning: sólo las últimas capas y con tasa 100 veces menor.
base.trainable=True
for capa in base.layers[:-20]: capa.trainable=False
modelo.compile(optimizer=keras.optimizers.Adam(1e-5),loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),metrics=["accuracy"])
modelo.fit(X_dev[tr],y_dev[tr],validation_data=(X_dev[val],y_dev[val]),epochs=3,batch_size=64,verbose=2)
pred=modelo.predict(X_test,batch_size=128,verbose=0).argmax(1)
print("accuracy test final:",accuracy_score(y_test,pred))


**Aplicación física propuesta:** sustituya CIFAR por imágenes de lentes
gravitacionales, cámaras de niebla o microscopía con licencia explícita.
Compare desde cero, extractor congelado y fine-tuning; controle que un
mismo objeto/experimento no aparezca en particiones distintas. Discuta el
cambio de dominio: las texturas de ImageNet no son leyes físicas.
